In [ ]:
# Import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,root_mean_squared_error,r2_score,mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
import joblib
import shap

In [ ]:

df = pd.read_csv('https://raw.githubusercontent.com/Ralshatiri/Car-Price-Prediction/refs/heads/main/Dataset/Final_processed_dataset.csv')
print(df.head(5))

           Make      Type      Price  Region  Gear_Type Origin    Options  \
0         Škoda    Superb   9.615805  Riyadh          0  Other  Semi Full   
1         Škoda    Superb   9.615805  Riyadh          1  Other  Semi Full   
2  Aston Martin       DB9  12.100712  Dammam          0  Saudi       Full   
3  Aston Martin       DB9  12.100712  Dammam          1  Saudi       Full   
4  Aston Martin  Vanquish  12.899220  Dammam          0  Saudi       Full   

   Engine_Size  Mileage  Negotiable  CarAge  
0          1.8   200000           0      18  
1          1.8   200000           0      18  
2          6.0    71000           0      16  
3          6.0    71000           0      16  
4          6.0    32000           0      13  


In [ ]:
x = df.drop('Price', axis=1)
y = df['Price']

In [ ]:
numerical_cols= ["Engine_Size", "Mileage", "CarAge"]
categorical_cols = ["Make", "Type", "Region", "Origin", "Options"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=0.2,random_state=42)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

In [ ]:
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

In [ ]:

encoded_cols = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
all_columns = numerical_cols + list(encoded_cols)
x_train_df = pd.DataFrame(x_train_processed, columns=all_columns)
print(x_train_df.head())

   Engine_Size   Mileage    CarAge  Make_Aston Martin  Make_Audi  Make_BMW  \
0    -1.119405 -0.996210 -0.667573                0.0        0.0       0.0   
1     1.342243  0.016727 -0.287688                0.0        0.0       0.0   
2     1.475305  1.061093  0.851967                0.0        0.0       0.0   
3     1.342243  0.282745 -0.287688                0.0        0.0       0.0   
4    -1.185936 -0.459149 -0.857516                0.0        0.0       0.0   

   Make_BYD  Make_Bentley  Make_Cadillac  Make_Changan  ...  Region_Taef  \
0       0.0           0.0            0.0           0.0  ...          0.0   
1       0.0           0.0            0.0           0.0  ...          0.0   
2       0.0           0.0            0.0           0.0  ...          0.0   
3       0.0           0.0            0.0           0.0  ...          0.0   
4       0.0           0.0            0.0           0.0  ...          0.0   

   Region_Wadi Dawasir  Region_Yanbu  Origin_Gulf Arabic  Origin_Other  \


In [ ]:
x_train_df.shape

(10338, 435)

In [ ]:
# Our Model
estimator = LinearRegression()
# select features
selector = RFECV(estimator,cv=5)
# find selected features
selector.fit(x_train_processed,y_train)
# print num of selected features
print("Optimal number of features: %d" % selector.n_features_)

Optimal number of features: 412


In [ ]:
# mask features
X_RFECV = x_train_processed[:,selector.support_]
# train model
estimator.fit(X_RFECV,y_train)

LinearRegression()

# **predictions are in log scale**

In [ ]:
# try model -> predictions are in log scale
y_pred_test = selector.predict(x_test_processed)
y_pred_train = selector.predict(x_train_processed)

In [ ]:
# convert predictions and actual values back to real price scale
y_pred_test_real = np.exp(y_pred_test)
y_pred_train_real = np.exp(y_pred_train)
y_test_real = np.exp(y_test)
y_train_real = np.exp(y_train)

print(y_pred_test_real)

[47447.89919703 44766.09632625 51382.31128518 ... 64503.71395053
 92050.66586611 81641.2716782 ]


In [ ]:
# MSE
MSE1 = mean_squared_error(y_test_real, y_pred_test_real)
print(MSE1)

# comparing MSE to find overfitting
MSE2 = mean_squared_error(y_train_real, y_pred_train_real)
print(MSE2)
print("ratio:", round(max(MSE2, MSE1) / min(MSE2, MSE1), 3))

# comparing RMSE(root MSE) to find overfitting
RMSE1 = root_mean_squared_error(y_test_real, y_pred_test_real)
RMSE2 = root_mean_squared_error(y_train_real, y_pred_train_real)
print("RMSE1: ", RMSE1)
print("RMSE2: ", RMSE2)
print("ratio:", round(max(RMSE2, RMSE1) / min(RMSE2, RMSE1), 3))

# prediction to labeled correctness metrics
# R2 score
r2 = r2_score(y_test_real, y_pred_test_real)
print(f"R^2 Score: {r2:.4f}")

# MAPE (Average percentage error)
mape = mean_absolute_percentage_error(y_test_real, y_pred_test_real)
print(f"Mean Absolute Percentage Error: {mape:.2f}")

1087597074.587852
1016723887.8419604
ratio: 1.07
RMSE1:  32978.7367039408
RMSE2:  31886.108069846974
ratio: 1.034
R^2 Score: 0.7675
Mean Absolute Percentage Error: 0.26


In [ ]:

joblib.dump(selector, "car_price_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

# **Reasoining**

In [ ]:
# SHAP explains which features influenced the model prediction
!pip install shap

In [ ]:

# training data with only selected features
X_selected_train = x_train_processed[:, selector.support_]

# test data with only selected features
X_selected_test = x_test_processed[:, selector.support_]

# names of selected features only
feature_names = [col for col, selected in zip(all_columns, selector.support_) if selected]

# create SHAP explainer using selected training features
explainer = shap.Explainer(selector.estimator_, X_selected_train)

# compute SHAP values using selected test features
shap_values = explainer(X_selected_test)

# predict log(price)
y_pred_log = selector.predict(x_test_processed)

# convert to real price
y_pred_price = np.exp(y_pred_log)

# show predicted price for first test car
print("Predicted Price:", round(y_pred_price[0], 2), "SAR")

# function to generate reasoning for a single prediction
def generate_reasoning(shap_values, feature_names, index=0):
    values = shap_values.values[index]
    reasoning = []

    for feature, value in zip(feature_names, values):
        if value > 0:
            reasoning.append(f"{feature} increased the predicted price")
        elif value < 0:
            reasoning.append(f"{feature} decreased the predicted price")

    return reasoning

# generate reasoning for first test car
reasoning = generate_reasoning(shap_values, feature_names, index=0)

print("\nReasoning for the prediction:")
for r in reasoning[:5]:
    print("-", r)

Predicted Price: 47447.9 SAR

Reasoning for the prediction:
- Mileage decreased the predicted price
- CarAge increased the predicted price
- Make_Audi decreased the predicted price
- Make_BYD increased the predicted price
- Make_Cadillac increased the predicted price


In [ ]:
X_selected_train = x_train_processed[:, selector.support_]
feature_names = [col for col, selected in zip(all_columns, selector.support_) if selected]

joblib.dump(X_selected_train[:200], "background_data.pkl")
joblib.dump(feature_names, "feature_names.pkl")

['feature_names.pkl']